# Notebook 2 — Booking Agent with BAML

In this notebook we'll build a parent-facing booking agent and progressively add reliability features. Each feature is motivated by a **demonstrated failure** — we'll see what goes wrong first, then add the fix.

By the end you'll have an agent that:

- runs a ReAct loop (thought → action → observation)
- uses multiple tools, but only the relevant ones (progressive disclosure)
- manages long conversations without bloat (context window + summarization)
- remembers parents across sessions (memory)
- refuses to leak nanny addresses pre-booking (permissions)
- gracefully degrades when tools fail (fallbacks)
- is deterministic by default (BAML schemas + temp=0)
- decomposes work across a Planner + Researcher + Executor (multi-agent)
- avoids the 5 multi-agent failure modes (handoff loss, telephone game, stale memory, role confusion, parallel disagreement)
- enforces input + output guardrails (PII redaction, safe escalation)

Every LLM call is **auto-traced to Phoenix** so you can inspect any of these runs in the Phoenix UI.

**Reads from Notebook 1:** `nanny_db/` (Chroma collection of nanny + parent profiles). If you skipped Notebook 1, re-run it now or rely on `data/seed_db.json` which ships pre-baked.

## 0. Setup + Phoenix instrumentation

Load `.env`, instrument Phoenix (so all subsequent OpenAI + BAML calls are auto-traced), and import the agent helpers we'll use throughout.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load .env BEFORE importing baml_client — BAML reads OPENAI_API_KEY at client-construct time.
load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY", "").startswith("sk-"), "Set OPENAI_API_KEY in .env"

from nanny_workshop.phoenix_setup import start_phoenix

phoenix_url, _ = start_phoenix()
print(f"📊 Phoenix UI: {phoenix_url}")
print("Every LLM call below is now auto-traced. Open the URL in a browser to watch.")

## 1. ReAct primer — single agent, one tool

The simplest agent: think → act → observe → think → ... → finish. Our BAML function `DecideOneTool` produces one structured `AgentStep` per turn. The Python `react_run` loop dispatches tools and feeds observations back.

Tool available this section: `search_nannies(query)` only.

In [ ]:
from baml_client.sync_client import b
from nanny_workshop.agent import react_run
from nanny_workshop.agent_tools import search_nannies

# Adapter: BAML's b.DecideOneTool returns an AgentStep object whose tool_call.args is a
# baml-typed map. react_run uses .tool_call.name, .tool_call.args (dict-like), .final_answer.
def decide_one_tool(user_message: str, history: str):
    return b.DecideOneTool(user_message=user_message, history=history)

trace = react_run(
    user_message="I need a CPR-certified nanny for Thursday mornings.",
    decide_fn=decide_one_tool,
    tools={"search_nannies": lambda query: search_nannies(query=query)},
    max_steps=4,
)

print(f"Steps taken: {len(trace.steps)}")
for i, s in enumerate(trace.steps, 1):
    print(f"\n--- step {i} ---")
    print(f"Thought: {s.thought}")
    print(f"Action:  {s.tool_name}({s.tool_args})")
    if s.observation is not None:
        obs_repr = str(s.observation)
        print(f"Observation: {obs_repr[:200]}{'...' if len(obs_repr) > 200 else ''}")
    if s.final_answer:
        print(f"Final: {s.final_answer}")

In [ ]:
# 🎯 TRY IT: change the user message and watch the agent's behavior.
#   - "Hi, I have a question about cancellation." — agent will likely try search_nannies first,
#     observe irrelevant results, then finish — showing why we need more than one tool.
#   - "I need a Spanish-speaking nanny who's available on weekends."
#   - Bump max_steps to 1 — what happens? (The agent never gets to finish.)
#
# Open the Phoenix UI from cell 3 and click into the trace. You'll see every LLM call
# with input/output, latency, and token counts.

your_query = "I need a Spanish-speaking nanny who's available on weekends."
trace = react_run(
    user_message=your_query,
    decide_fn=decide_one_tool,
    tools={"search_nannies": lambda query: search_nannies(query=query)},
    max_steps=4,
)
print(trace.final_answer)